# Part 4 · Recycling Instruction Generation (RAG)
**Owner: Eglen**

### Setup (VS Code / local) — run first

In [ ]:
import os, zipfile, pathlib, shutil
# Unzip data/realwaste.zip -> data/RealWaste/<category>/ if not already done
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z: z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p),"data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)
IMAGE_DIR = "data/RealWaste"
print("Images present:", os.path.isdir(IMAGE_DIR))

In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

### 4.1 Build retrieval corpus (policy docs + disposal tips)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
import faiss
with open("data/waste_policy_documents.json") as f: policies = json.load(f)
desc = pd.read_csv("data/waste_descriptions.csv")
corpus=[]
for p in policies:
    corpus.append({"source":p["policy_type"],"categories":p.get("categories_covered",[]),"text":p["document_text"]})
for cat,sub in desc.groupby("category"):
    tips=sorted(set(sub["disposal_instruction"].dropna()))[:5]
    corpus.append({"source":f"{cat} disposal tips","categories":[cat],
                   "text":f"Disposal guidance for {cat}: "+" ".join(tips)})
print(len(corpus),"chunks")

### 4.2 Embed + index + generator

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb = embedder.encode([c["text"] for c in corpus], normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
gen = hf_pipeline("text2text-generation", model="google/flan-t5-base")

### 4.3 Retrieve + generate (grader interface)

In [ ]:
def retrieve(query, k=3, category=None):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index.search(q, k*3)
    hits=[corpus[i] for i in idx[0]]
    if category: hits=sorted(hits, key=lambda d: category not in d["categories"])
    return hits[:k]

def generate_recycling_instructions(waste_category):
    docs = retrieve(f"How to recycle {waste_category}?", k=3, category=waste_category)
    context = "\n\n".join(f"[{d['source']}]\n{d['text']}" for d in docs)
    prompt = ("You are a city recycling assistant. Using ONLY the policy context below, give clear "
              f"step-by-step recycling instructions for '{waste_category}'.\n\nContext:\n{context}\n\nInstructions:")
    text = gen(prompt, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]
    return text, docs

In [ ]:
for cat in ["Plastic","Metal","Glass"]:
    t,d = generate_recycling_instructions(cat)
    print(cat,"->",t[:250]); print("  sources:",[x["source"] for x in d],"\n")

### TODO (Eglen)\n- Experiment with temperature / top_p / max_new_tokens\n- Evaluate factual consistency vs retrieved policy text\n- Comment on readability (grounding is the point)